<a href="https://colab.research.google.com/github/LOSTHADY/IA_Project-/blob/claude%2Fllm-bambara-audio-v7xgtr/chat-ia-bambara-dho8oa/notebooks/phase3_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phases 3 et 4 — fine-tuning, puis cascade contre bout-en-bout

1. **Whisper multi-tâche** sur Jeli-ASR (`--task both`) : un seul modèle,
   un seul encodeur, deux sorties. `<|transcribe|>` donne le bambara (la
   cascade), `<|translate|>` le français (le bout-en-bout).
2. **NLLB**, un modèle par sens, sur Bayelemabaga.
3. **Évaluation** avec le protocole de la phase 1 (même partition, même
   graine, mêmes 300 énoncés), contre les références non affinées
   recalculées ici, dans le même environnement.
4. **Test de l'assistant affiné** : la chaîne complète, puis la démo
   vocale avec un lien pour lui parler au micro.

**Durée.** En `--task both`, une époque de Whisper fait environ 64 000
exemples (deux par enregistrement), soit environ 4 000 étapes. Sur un T4,
c'est plusieurs heures : l'essai rapide (§4) mesure la vitesse réelle et
annonce la durée avant de lancer quoi que ce soit de long. Une session
Colab gratuite sera probablement coupée en route : les checkpoints sont
écrits sur Drive, et `--resume` repart du dernier. Après une coupure,
relancer les cellules 1 à 3, puis la cellule d'entraînement interrompue.

*Exécution → Modifier le type d'exécution → GPU (T4).*

## 1. Installation

In [1]:
# Branche de travail : contient les dernières corrections du notebook et des
# scripts. Une fois fusionnée, la branche par défaut suffit (sans -b).
BRANCH = "claude/chat-ia-bambara-dho8oa"

!git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

/content/projet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.2 MB/s eta 0:00:00


## 2. Dossiers sur Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/bambara/modeles"            # checkpoints et modèles finaux
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"    # rapports d'évaluation
!mkdir -p "$CKPT" "$OUT"

Mounted at /content/drive


## 3. Accès aux jeux et sous-ensemble Jeli-ASR

Même réglage qu'en phase 1. Si Jeli-ASR demande un sous-ensemble, reprendre
celui choisi alors : les chiffres ne se comparent que sur le même jeu.

In [3]:
JELI_CONFIG = ""  # ex. "bam-asr-oza", comme en phase 1

JELI = "--dataset RobotsMali/jeli-asr" + (f" --dataset-config {JELI_CONFIG}" if JELI_CONFIG else "")
JELI_EVAL = "--dataset RobotsMali/jeli-asr" + (f" --config {JELI_CONFIG}" if JELI_CONFIG else "")

# Si un jeu est « gated » : décommenter.
# from huggingface_hub import notebook_login; notebook_login()

# Époques d'entraînement. Une est un bon premier passage sur Colab gratuit ;
# l'essai du §4 annonce la durée de chaque option avant de lancer le long.
EPOCHS_WHISPER = 1
EPOCHS_NLLB = 1

## 4. Essai rapide (10 minutes) et durée prévue

400 enregistrements, une époque, soit une cinquantaine d'étapes. Vérifie
que tout s'enchaîne (données, colonnes, mémoire GPU) et mesure la vitesse
de ce GPU. Les scores de cet essai ne veulent rien dire.

In [4]:
!python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
    --max-samples 400 --epochs 1 --save-steps 50 --eval-samples 20 --output /content/essai

preprocessor_config.json: 100% 185k/185k [00:00<00:00, 296MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 7.54MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 250MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 109MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 148MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 104MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 104MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 77.4MB/s]
special_tokens_map.json: 100% 2.19k/2.19k [00:00<00:00, 9.58MB/s]

model.safetensors: downloading bytes:   9% 86.0M/967M [00:01<00:10, 87.8MB/s, 6.10MB/s  ]
model.safetensors: downloading bytes:  18% 171M/967M [00:02<00:06, 120MB/s, 15.0MB/s  ]
model.safetensors: downloading bytes:  20% 193M/967M [00:02<00:06, 115MB/s, 16.6MB/s  ]
model.safetensors: downloading bytes:  50% 486M/967M [00:04<00:02, 228MB/s, 38.5MB/s  ]
model.safetensors: reconstructing file:  33% 321M/967M [00:04<00:08, 72.8MB/s, 24.9MB/s  ]
model.safetensors: downloading bytes

In [5]:
# Durée prévue de l'entraînement complet, d'après la vitesse de l'essai.
import json, math
state = json.load(open("/content/essai/trainer_state.json"))  # écrit en fin d'essai
last = [h for h in state["log_history"] if "train_runtime" in h][-1]
s_per_step = last["train_runtime"] / state["global_step"]
steps_per_epoch = math.ceil(2 * 32_000 / (8 * 2))   # 2 tâches, lot 8 × accumulation 2
print(f"{s_per_step:.2f} s par étape (essai, démarrage compris : plutôt pessimiste)")
for epochs in (1, 2, 3):
    hours = epochs * steps_per_epoch * s_per_step * 1.1 / 3600   # +10 % d'évaluations
    print(f"Whisper, {epochs} époque(s) : environ {hours:.1f} h")

print("Réglage actuel (§3) :", EPOCHS_WHISPER, "époque(s) pour Whisper,", EPOCHS_NLLB, "pour NLLB")

3.80 s par étape (essai, démarrage compris : plutôt pessimiste)
Whisper, 1 époque(s) : environ 4.6 h
Whisper, 2 époque(s) : environ 9.3 h
Whisper, 3 époque(s) : environ 13.9 h
Réglage actuel (§3) : 1 époque(s) pour Whisper, 1 pour NLLB


## 5. Fine-tuning de Whisper (long)

Le meilleur checkpoint est choisi au WER sur une validation prise dans
`train`. La partition de test reste intacte pour l'évaluation (§7).
Relancer cette cellule après une coupure : elle reprend au dernier
checkpoint.

In [ ]:
!python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
    --epochs $EPOCHS_WHISPER --output "$CKPT/whisper-small-bm" --resume

Loading weights: 100% 479/479 [00:00<00:00, 789.01it/s]
Schéma : {'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'duration': Value('float32'), 'bam': Value('string'), 'french': Value('string')}
Colonnes : bambara=bam, français=french, audio=audio
Entraînement : 60483 exemples (transcribe + translate) ; validation : 298 (transcribe, partition prise dans train)
Aucun checkpoint : départ de zéro
{'loss': '9.127', 'grad_norm': '66.9', 'learning_rate': '8e-07', 'epoch': '0.006613'}
{'loss': '7.759', 'grad_norm': '43.95', 'learning_rate': '1.633e-06', 'epoch': '0.01323'}
{'loss': '6.27', 'grad_norm': '33.18', 'learning_rate': '2.467e-06', 'epoch': '0.01984'}
{'loss': '5.381', 'grad_norm': '28.23', 'learning_rate': '3.3e-06', 'epoch': '0.02645'}
{'loss': '4.886', 'grad_norm': '33.54', 'learning_rate': '4.133e-06', 'epoch': '0.03306'}
{'loss': '4.495', 'grad_norm': '28.74', 'learning_rate': '4.967e-06', 'epoch': '0.03968'}
{'loss': '4.193', 'grad_norm':

## 6. Fine-tuning de NLLB, un modèle par sens

In [ ]:
!python scripts/finetune_nllb.py --direction bm2fr --epochs $EPOCHS_NLLB \
    --output "$CKPT/nllb-bm2fr" --resume

In [ ]:
!python scripts/finetune_nllb.py --direction fr2bm --epochs $EPOCHS_NLLB \
    --output "$CKPT/nllb-fr2bm" --resume

## 7. Évaluation — cascade contre bout-en-bout

Même partition, même graine et même nombre d'énoncés qu'en phase 1. Les
deux architectures utilisent **le même Whisper** : seule la tâche change.

### Références non affinées, dans le même environnement

Les mêmes mesures qu'en phase 1 (`docs/RESULTATS.md`), recalculées ici sur
les mêmes énoncés, pour que les écarts avec les modèles affinés se testent
phrase par phrase. Une vingtaine de minutes sur GPU.

In [ ]:
!python -m eval.baselines --out "$OUT" --label "MMS + NLLB zero-shot" \
    asr $JELI_EVAL --model facebook/mms-1b-all --kind ctc --target-lang bam --with-mt --limit 300
!python -m eval.baselines --out "$OUT" --label "NLLB zero-shot / Bayelemabaga" \
    mt --dataset RobotsMaliAI/bayelemabaga --limit 500
!python -m eval.baselines --out "$OUT" --label "NLLB zero-shot / Jeli-ASR" \
    mt $JELI_EVAL --limit 500

In [ ]:
!python -m eval.baselines --out "$OUT" --label "cascade affinée" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" \
    --with-mt --mt-model "$CKPT/nllb-bm2fr" --limit 300

In [ ]:
!python -m eval.baselines --out "$OUT" --label "bout-en-bout affiné" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" --task translate --limit 300

Pour séparer l'apport de chaque fine-tuning : la cascade avec le Whisper
affiné mais le NLLB d'origine. Facultatif.

In [ ]:
!python -m eval.baselines --out "$OUT" --label "cascade, NLLB d'origine" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" --with-mt --limit 300

### Traduction seule, dans les deux sens

In [ ]:
!python -m eval.baselines --out "$OUT" --label "NLLB affiné / Bayelemabaga" \
    mt --dataset RobotsMaliAI/bayelemabaga --limit 500 \
    --bm2fr-model "$CKPT/nllb-bm2fr" --fr2bm-model "$CKPT/nllb-fr2bm"

In [ ]:
!python -m eval.baselines --out "$OUT" --label "NLLB affiné / Jeli-ASR" \
    mt $JELI_EVAL --limit 500 \
    --bm2fr-model "$CKPT/nllb-bm2fr" --fr2bm-model "$CKPT/nllb-fr2bm"

## 8. Tableau commun : références non affinées et modèles affinés

In [ ]:
!python -m eval.compare "$OUT"/*.json

### L'écart entre les architectures est-il réel ?

Intervalles de confiance à 95 % et test apparié par bootstrap, sur les mêmes
énoncés. Si l'IC de l'écart contient zéro, la différence n'est pas établie :
c'est ce qu'il faut écrire dans le mémoire, quelle que soit la taille
apparente de l'écart.

In [ ]:
# Ce que le fine-tuning apporte à la cascade (WER et chrF++, mêmes énoncés)
!python -m eval.significance "$OUT"/asr-*.json --noms "MMS + NLLB zero-shot" "cascade affinée"
# La comparaison centrale : cascade contre bout-en-bout (chrF++ depuis l'audio)
!python -m eval.significance "$OUT"/asr-*.json --noms "cascade affinée" "bout-en-bout affiné"
# Traduction seule, avant et après fine-tuning
!python -m eval.significance "$OUT"/mt-*.json --noms "NLLB zero-shot / Bayelemabaga" "NLLB affiné / Bayelemabaga"

### Comment les modèles affinés se trompent-ils ?

Même analyse d'erreurs que pour la phase 1 (`docs/RESULTATS.md`). À
vérifier en premier : le mot swahili « kwa » et les boucles de répétition,
qui expliquaient l'essentiel des insertions de Whisper zero-shot, doivent
avoir disparu. Le mot « kwa » trahirait un token de langue mal réaffecté
au bambara.

In [ ]:
!python -m eval.analysis "$OUT"/*.json | tee "$OUT/analyse.md"

## Lire les résultats

- **La comparaison centrale** est la ligne *chrF++ (depuis ASR)* : le
  français obtenu par la cascade (ASR puis NLLB) contre celui que Whisper
  produit directement, sur les mêmes énoncés et la même référence.
- **Propagation d'erreurs** (cascade) : ce que les erreurs d'ASR coûtent à
  la traduction. C'est le coût que le bout-en-bout cherche à supprimer.
  S'il reste faible, le maillon supplémentaire de la cascade est peu
  pénalisant, et elle garde ses avantages : texte bambara affichable et
  gabarits utilisables.
- **Latence** : mesurée sur GPU, donc indicative seulement. Le budget de
  déploiement se mesure sur CPU (phase 5).
- **Limites à écrire dans le mémoire** : Jeli-ASR est à la fois la source
  d'entraînement et d'évaluation. Si des locuteurs de test apparaissent
  aussi dans `train` (à vérifier dans la documentation du jeu), les scores
  sont optimistes. C'est pourquoi les chiffres définitifs se mesurent sur le
  jeu de test maison (phase 2), avec `eval.run_eval`.

## 9. Tester l'assistant affiné

La chaîne complète avec les modèles affinés : reconnaissance, traduction,
LLM (Qwen2.5-1.5B, ouvert), traduction, synthèse. D'abord sur 20 énoncés de
Jeli-ASR, dans les deux architectures, avec le temps de chaque étape et les
réponses produites ; puis en direct, au micro.

In [ ]:
import json
CFG = "/content/assistant-affine.json"
json.dump({"device": "cuda",
           "asr": {"model_id": f"{CKPT}/whisper-small-bm"},
           "mt_in": {"model_id": f"{CKPT}/nllb-bm2fr"},
           "mt_out": {"model_id": f"{CKPT}/nllb-fr2bm"}}, open(CFG, "w"), indent=1)
!python -m bambara_voice.cli --config $CFG check
!python scripts/export_corpus_sample.py $JELI_EVAL --limit 20 --out /content/echantillon

In [ ]:
!python -m eval.run_eval --testset /content/echantillon/testset.jsonl --arch cascade \
    --config $CFG --synthesize --out "$OUT/chaine" > /dev/null
!python -m eval.run_eval --testset /content/echantillon/testset.jsonl --arch e2e \
    --config $CFG --synthesize --out "$OUT/chaine" > /dev/null

import glob
for path in sorted(glob.glob(f"{OUT}/chaine/*-*[0-9].json")):
    r = json.load(open(path))
    rows = [json.loads(l) for l in open(path.replace(".json", ".details.jsonl"))]
    print(f"\n=== {r['architecture']} — {r['latence']['total_moyen_s']:.1f} s par tour :",
          r["latence"]["par_etape_s"])
    for x in rows[:8]:
        print(f"- dit : {x['ref_transcript_bm']}\n  compris : {x['source_bm'] or '—'}"
              f" | {x['source_fr']}\n  réponse : {x['reply_fr']}\n  en bambara : {x['reply_bm']}")

### Au micro

La cellule suivante affiche un lien `https://….gradio.live` : l'ouvrir, autoriser
le micro, parler en bambara. Elle tourne jusqu'à ce qu'on l'arrête (bouton
stop). Pour la variante bout-en-bout, ajouter `--arch e2e`.

In [ ]:
!pip install -q gradio
!python app/gradio_app.py --config $CFG --share

## 10. Préparer le déploiement sur CPU (phase 5)

Convertit les trois modèles en int8 (CTranslate2) sur Drive, avec le
fichier `config.json` qui les décrit. Télécharger ensuite le dossier
`modeles-cpu` sur la machine de déploiement : voir `docs/DEPLOIEMENT.md`
pour le LLM en GGUF, la perte de qualité à mesurer et la latence.

In [ ]:
!pip install -q ctranslate2
!python scripts/export_cpu.py --out "$CKPT/../modeles-cpu" \
    --whisper "$CKPT/whisper-small-bm" \
    --nllb-bm2fr "$CKPT/nllb-bm2fr" --nllb-fr2bm "$CKPT/nllb-fr2bm"

## Facultatif — publier les modèles (en privé)

Pour les réutiliser ailleurs (démo, phase 5) sans passer par Drive.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# for name in ("whisper-small-bm", "nllb-bm2fr", "nllb-fr2bm"):
#     repo = api.create_repo(f"VOTRE_COMPTE/{name}", private=True, exist_ok=True).repo_id
#     api.upload_folder(folder_path=f"{CKPT}/{name}", repo_id=repo,
#                       ignore_patterns=["checkpoint-*"])